# Исследование влияния миграции на динамику эпидемии

**Цель:** Изучить, как интенсивность перемещения людей между городами влияет
на скорость распространения эпидемии (время достижения пика) и масштаб пика.
Инфекция начинается только в одном городе, остальные изначально здоровы.

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots, CSV, Random

Подключение модуля с определением модели SIR

In [ ]:
include(srcdir("sir_model.jl"))

## Вспомогательные функции

### Создание матрицы миграции

Функция создаёт матрицу вероятностей миграции между городами на основе
заданной интенсивности. При интенсивности `intensity` = 0 миграция отсутствует,
при `intensity` = 0.5 вероятность остаться в городе равна 0.5, а вероятности
переезда в другие города распределены равномерно.

In [ ]:
function create_migration_matrix(C, intensity)

C — количество городов
intensity — интенсивность миграции (вероятность покинуть город)

Создаём матрицу C×C, где все элементы равны intensity/(C-1)

In [ ]:
    M = ones(C, C) .* intensity ./ (C-1)

Заполняем диагональ: вероятность остаться в своём городе

In [ ]:
    for i = 1:C
        M[i, i] = 1 - intensity
    end

    return M
end

### Измерение времени достижения пика эпидемии

Функция запускает симуляцию с заданными параметрами и определяет момент времени,
когда доля инфицированных достигает максимального значения.

In [ ]:
function peak_time(p)

Создаём матрицу миграции на основе интенсивности

In [ ]:
    migration_rates = create_migration_matrix(p[:C], p[:migration_intensity])

Инициализируем модель с переданными параметрами

In [ ]:
    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = p[:β_und],
        β_det = p[:β_det],
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
        migration_rates = migration_rates,
    )

Вспомогательная функция для вычисления доли инфицированных

In [ ]:
    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)

    peak = 0.0      # максимальное значение доли инфицированных
    peak_step = 0   # шаг времени, на котором достигнут пик

Основной цикл симуляции

In [ ]:
    for step = 1:p[:n_steps]

Ручной шаг моделирования (обход всех агентов)

In [ ]:
        agent_ids = collect(allids(model))
        for id in agent_ids
            agent = try
                model[id]
            catch
                nothing
            end
            if agent !== nothing
                sir_agent_step!(agent, model)
            end
        end

Вычисляем текущую долю инфицированных

In [ ]:
        frac = infected_frac(model)

Обновляем информацию о пике

In [ ]:
        if frac > peak
            peak = frac
            peak_step = step
        end
    end

    return (peak_time = peak_step, peak_value = peak)
end

## Параметрическое сканирование

Исследуем влияние интенсивности миграции на динамику эпидемии.
Будем варьировать интенсивность от 0 до 0.5 с шагом 0.1.

Диапазон значений интенсивности миграции

In [ ]:
migration_intensities = 0.0:0.1:0.5

Набор случайных зёрен для воспроизводимости результатов

In [ ]:
seeds = [42, 43, 44]

### Формирование списка параметров

Создаём все возможные комбинации параметров для экспериментов.

In [ ]:
params_list = []

for mig in migration_intensities
    for s in seeds
        push!(
            params_list,
            Dict(
                :migration_intensity => mig,              # скалярное значение интенсивности
                :C => 3,                                  # количество городов
                :Ns => [1000, 1000, 1000],               # численность населения в городах
                :β_und => [0.5, 0.5, 0.5],               # заразность невыявленных
                :β_det => [0.05, 0.05, 0.05],            # заразность выявленных
                :infection_period => 14,                 # длительность болезни (дни)
                :detection_time => 7,                    # время до выявления (дни)
                :death_rate => 0.02,                     # вероятность смерти
                :reinfection_probability => 0.1,         # вероятность повторного заражения
                :Is => [1, 0, 0],                        # начальные заражённые (только в городе 1)
                :seed => s,                              # зерно генератора случайных чисел
                :n_steps => 150,                         # количество дней симуляции
            ),
        )
    end
end

### Запуск экспериментов

Выполняем симуляции для всех комбинаций параметров.

In [ ]:
results = []

for params in params_list
    data = peak_time(params)
    push!(results, merge(params, Dict(pairs(data))))
    println(
        "Завершён эксперимент с migration_intensity = $(params[:migration_intensity]), seed = $(params[:seed])",
    )
end

## Сохранение результатов

Сохраняем данные всех прогонов в CSV-файл для последующего анализа.

In [ ]:
df = DataFrame(results)
CSV.write(datadir("migration_scan_all.csv"), df)

### Усреднение по повторным прогонам

Для каждого значения интенсивности миграции усредняем результаты
по трём различным случайным зёрнам.

In [ ]:
using Statistics

grouped = combine(
    groupby(df, [:migration_intensity]),
    :peak_time => mean => :mean_peak_time,
    :peak_value => mean => :mean_peak_value,
)

## Визуализация результатов

Строим график зависимости времени достижения пика и пиковой заболеваемости
от интенсивности миграции.

График времени до пика

In [ ]:
plot(
    grouped.migration_intensity,
    grouped.mean_peak_time,
    marker = :circle,
    xlabel = "Интенсивность миграции",
    ylabel = "Время до пика (дни)",
    label = "Время пика",
)

Наложение графика пиковой заболеваемости

In [ ]:
plot!(
    grouped.migration_intensity,
    grouped.mean_peak_value .* 3000,  # умножаем на общую численность населения (3000)
    marker = :square,
    xlabel = "Интенсивность миграции",
    ylabel = "Численность в пике",
    label = "Пиковая заболеваемость",
)

Сохраняем график

In [ ]:
savefig(plotsdir("migration_effect.png"))

## Вывод информации о результатах

In [ ]:
println("Результаты сохранены в data/migration_scan_all.csv и plots/migration_effect.png")

## Интерпретация результатов

График демонстрирует, как ускорение обмена людьми между городами приводит
к более раннему и более высокому пику эпидемии:

- **Время до пика** уменьшается с ростом интенсивности миграции — инфекция
  быстрее распространяется между городами.
- **Пиковая заболеваемость** увеличивается — больше людей оказываются
  инфицированными одновременно, что создаёт повышенную нагрузку на систему
  здравоохранения.